# TP5 - Punto 2
## Crecimiento poblacional de Malthus

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.integrate import solve_ivp
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4.8)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def resolver_edo_escalar(f, y0, t_final, n=500, rtol=1e-9, atol=1e-12):
    """Resuelve una EDO escalar y devuelve t, y."""
    t_eval = np.linspace(0, t_final, n)
    sol = solve_ivp(
        fun=lambda t, y: [f(t, y[0])],
        t_span=(0, t_final),
        y0=[y0],
        t_eval=t_eval,
        rtol=rtol,
        atol=atol,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol.t, sol.y[0]

def valores_en(t, y, puntos, nombre="valor"):
    return pd.DataFrame({
        "t": puntos,
        nombre: np.interp(puntos, t, y),
    })

def primer_tiempo_cerca(t, y, objetivo, tolerancia):
    cerca = np.where(np.abs(y - objetivo) <= tolerancia)[0]
    if len(cerca) == 0:
        return np.nan
    return t[cerca[0]]

def mostrar_tabla(df, decimales=4):
    display(df.round(decimales).style.hide(axis="index"))

## 2. Crecimiento poblacional de Malthus

Modelo:

`dP/dt = r P`

En esta seccion el tiempo `t` se mide en años y `r` tiene unidades de `1/año`. Igual que pasaba con `k` en Newton, el valor numerico de `r` depende de la unidad de tiempo elegida.

`r` representa la tasa neta de crecimiento: tasa de natalidad menos tasa de mortalidad por individuo y por unidad de tiempo. Por eso, `r > 0` significa que nacen mas individuos de los que mueren.

Clasificacion por el signo de `r`:

- `r > 0`: la poblacion crece exponencialmente.
- `r = 0`: la poblacion queda constante.
- `r < 0`: la poblacion decrece hacia `0`.

Para `r > 0`, el tiempo de duplicacion es `ln(2)/r`. Ese tiempo indica cuanto tarda la poblacion en multiplicarse por `2`.

In [ ]:
def malthus_rhs(r):
    return lambda t, P: r * P

def resolver_malthus(P0=100, r=0.15, t_final=20):
    return resolver_edo_escalar(malthus_rhs(r), P0, t_final)

def malthus_teorica(t, P0, r):
    return P0 * np.exp(r * t)

P0 = 100
r = 0.15
t, P = resolver_malthus(P0=P0, r=r, t_final=20)
P_exacta = malthus_teorica(t, P0, r)
tiempo_duplicacion = np.log(2) / r

print(f"Caso base: P0={P0}, r={r} 1/año")
print(f"Tiempo de duplicacion: {tiempo_duplicacion:.4f} años")
print(f"Error maximo contra la formula cerrada: {np.max(np.abs(P - P_exacta)):.2e}")
mostrar_tabla(valores_en(t, P, [0, 5, 10, 20, tiempo_duplicacion], "P(t)"))

plt.plot(t, P, label="solve_ivp", linewidth=2)
plt.plot(t, P_exacta, "--", label="formula cerrada", linewidth=1.6)
plt.title("Malthus - caso base")
plt.xlabel("Tiempo (años)")
plt.ylabel("Poblacion")
plt.legend()
plt.show()

### Variar `r` con `P0` fijo

Se deja `P0 = 100` fijo y se cambia `r`. En los graficos, `P0` se marca con una linea y un punto amarillo para recordar que todas las curvas parten de la misma poblacion inicial.

Como `r = 0.30` crece muy rapido, se separan los casos `r >= 0` y `r < 0` para que las curvas no se aplasten.

Tambien se muestra el caso `r >= 0` con eje vertical logaritmico. En escala logaritmica, una exponencial se ve como una recta. La pendiente de esa recta depende de `r`: cuanto mayor es `r`, mas inclinada aparece.

En la tabla, `P despues de 20 años` es la poblacion que predice el modelo al final de la simulacion, usando `t = 20`. Si `r = 0`, la poblacion no crece y por eso no existe tiempo de duplicacion: se indica como `no se duplica`.

In [ ]:
P0 = 100
r_valores_no_negativos = [0.00, 0.08, 0.15, 0.30]
r_valores_negativos = [-0.20, -0.05]
resumen = []

plt.figure(figsize=(9, 4.8))
plt.axhline(P0, color="gold", linestyle="--", linewidth=2.2, label=f"P0 inicial = {P0}")
plt.scatter([0], [P0], color="gold", edgecolor="black", s=60, zorder=5)
for ri in r_valores_no_negativos:
    t, P = resolver_malthus(P0=P0, r=ri, t_final=20)
    plt.plot(t, P, linewidth=2, label=f"r={ri}")
    resumen.append({
        "r": ri,
        "P despues de 20 años": P[-1],
        "tipo": "constante" if ri == 0 else "crece",
        "tiempo de duplicacion": "no se duplica" if ri <= 0 else np.log(2) / ri,
        "interpretacion": "sin crecimiento" if ri == 0 else "crece mas rapido cuanto mayor es r",
    })
plt.title(f"Malthus - variar r con P0 fijo (P0={P0}, r >= 0)")
plt.xlabel("Tiempo (años)")
plt.ylabel("Poblacion")
plt.legend()
plt.show()

plt.figure(figsize=(9, 4.8))
plt.axhline(P0, color="gold", linestyle="--", linewidth=2.2, label=f"P0 inicial = {P0}")
plt.scatter([0], [P0], color="gold", edgecolor="black", s=60, zorder=5)
for ri in r_valores_no_negativos:
    t, P = resolver_malthus(P0=P0, r=ri, t_final=20)
    plt.plot(t, P, linewidth=2, label=f"r={ri}")
plt.yscale("log")
plt.title(f"Malthus - r >= 0 con eje y logaritmico (P0={P0})")
plt.xlabel("Tiempo (años)")
plt.ylabel("Poblacion (escala log)")
plt.legend()
plt.show()

plt.figure(figsize=(9, 4.8))
plt.axhline(P0, color="gold", linestyle="--", linewidth=2.2, label=f"P0 inicial = {P0}")
plt.scatter([0], [P0], color="gold", edgecolor="black", s=60, zorder=5)
for ri in r_valores_negativos:
    t, P = resolver_malthus(P0=P0, r=ri, t_final=40)
    plt.plot(t, P, linewidth=2, label=f"r={ri}")
    resumen.append({
        "r": ri,
        "P despues de 20 años": np.interp(20, t, P),
        "tipo": "decrece",
        "tiempo de duplicacion": "no se duplica",
        "interpretacion": "decrece hacia 0 sin llegar exactamente",
    })
plt.axhline(1, color="tab:red", linestyle=":", label="P=1")
plt.title(f"Malthus - variar r con P0 fijo (P0={P0}, r < 0)")
plt.xlabel("Tiempo (años)")
plt.ylabel("Poblacion")
plt.legend()
plt.show()

mostrar_tabla(pd.DataFrame(resumen))

### Variar `P0` con `r` fijo

Ahora se deja `r = 0.15 1/año` fijo y se cambia `P0`. El tiempo de duplicacion `ln(2)/r` es el mismo para cualquier `P0` positivo.

Para que el grafico no quede aplastado por `P0 = 1000`, se separan los valores chicos en un panel y `P0 = 1000` en otro. En ambos paneles se marca `P = 0` y `t = 0` para que el origen se vea claro.

La tabla compara la poblacion calculada en el tiempo de duplicacion contra `2*P0`. Si coinciden, significa que la poblacion se duplico.

In [ ]:
r = 0.15
P0_valores = [1, 50, 100, 1000]
tiempo_duplicacion = np.log(2) / r
resumen = []

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharex=True)
paneles = [
    (axes[0], [1, 50, 100], "P0 chicos y mediaños"),
    (axes[1], [1000], "P0 grande"),
]

for ax, valores_panel, titulo_panel in paneles:
    ax.axhline(0, color="black", linewidth=1.2)
    ax.axvline(0, color="black", linewidth=1.2)
    ax.axvline(tiempo_duplicacion, color="tab:red", linestyle=":", linewidth=1.8, label=f"t duplicacion = {tiempo_duplicacion:.2f} años")

    for P0i in valores_panel:
        t, P = resolver_malthus(P0=P0i, r=r, t_final=20)
        P_en_duplicacion = np.interp(tiempo_duplicacion, t, P)
        ax.plot(t, P, linewidth=2, label=f"P0={P0i}")
        ax.scatter([0], [P0i], s=45, zorder=5)

        resumen.append({
            "P0 inicial": P0i,
            "r fijo": r,
            "tiempo para duplicarse (años)": tiempo_duplicacion,
            "2 * P0": 2 * P0i,
            "P calculada en ese tiempo": P_en_duplicacion,
            "interpretacion": "coincide con 2*P0: se duplico",
        })

    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.set_title(titulo_panel)
    ax.set_xlabel("Tiempo (años)")
    ax.set_ylabel("Poblacion")
    ax.legend()

fig.suptitle(f"Malthus - variar P0 con r fijo (r={r} 1/año)")
plt.tight_layout()
plt.show()

tabla = pd.DataFrame(resumen).sort_values("P0 inicial").round(4)
display(tabla.style.hide(axis="index"))

**Interpretacion del resultado.**

Este bloque muestra que, si `r` queda fijo, cambiar `P0` solo cambia la escala de la poblacion. Las curvas arrancan en valores distintos, pero todas se duplican en el mismo tiempo.

Con `r = 0.15 1/año`, el tiempo de duplicacion es `ln(2)/r = 4.62 años`. Por eso, una poblacion que empieza en `1` llega a `2`, una que empieza en `50` llega a `100`, una que empieza en `100` llega a `200` y una que empieza en `1000` llega a `2000` en el mismo tiempo.

Esto pasa porque la velocidad de crecimiento es proporcional a la poblacion actual. Una poblacion grande suma mas individuos por año, pero tambien necesita sumar mas individuos para duplicarse. En proporcion, todas crecen al mismo ritmo.

Entonces, `r` define el ritmo relativo del crecimiento y `P0` solo define el punto de partida. Esta es la misma idea que vimos en Newton con el tiempo mitad y en decaimiento radioactivo con la vida media.

La limitacion es que el modelo trata la poblacion como una cantidad continua. Para poblaciones chicas, como `P0 = 1`, la interpretacion biologica es debil: un solo individuo no se reproduce solo. El modelo tiene mas sentido para poblaciones grandes.

### Casos especiales

Esta tabla resume casos borde del modelo. Sirve para separar lo que la ecuacion puede calcular de lo que tiene sentido biologico.

In [ ]:
casos_malthus = [
    {"caso": "r = 0", "P0": 100, "r": 0, "t_final": 20},
    {"caso": "r < 0", "P0": 100, "r": -0.15, "t_final": 40},
    {"caso": "P0 = 0", "P0": 0, "r": 0.15, "t_final": 20},
    {"caso": "P0 < 0", "P0": -100, "r": 0.15, "t_final": 20},
    {"caso": "P0 = 1", "P0": 1, "r": 0.15, "t_final": 20},
    {"caso": "poblacion no entera", "P0": 100, "r": 0.15, "t_final": 5},
    {"caso": "r muy grande", "P0": 100, "r": 0.60, "t_final": 20},
]

filas = []
for caso in casos_malthus:
    t, P = resolver_malthus(P0=caso["P0"], r=caso["r"], t_final=caso["t_final"])
    tiempo_menor_1 = "no aplica"
    if caso["r"] < 0 and caso["P0"] > 1:
        tiempo_menor_1 = np.log(1 / caso["P0"]) / caso["r"]

    filas.append({
        "caso": caso["caso"],
        "P0 inicial": caso["P0"],
        "r (1/año)": caso["r"],
        "tiempo simulado": caso["t_final"],
        "P al final de la simulacion": P[-1],
        "tiempo hasta P < 1": tiempo_menor_1,
        "interpretacion": (
            "equilibrio constante" if caso["r"] == 0 else
            "decrece pero nunca llega a 0; en la realidad se extingue cuando P < 1" if caso["r"] < 0 else
            "solucion nula" if caso["P0"] == 0 else
            "no fisico: poblacion negativa" if caso["P0"] < 0 else
            "un solo individuo no se reproduce; el modelo es continuo" if caso["caso"] == "P0 = 1" else
            "aparecen valores no enteros; aproximacion continua para poblaciones grandes" if caso["caso"] == "poblacion no entera" else
            "crecimiento explosivo del modelo ideal"
        ),
    })

tabla = pd.DataFrame(filas)
columnas_numericas = ["r (1/año)", "P al final de la simulacion"]
tabla[columnas_numericas] = tabla[columnas_numericas].round(4)
tabla["tiempo hasta P < 1"] = tabla["tiempo hasta P < 1"].apply(lambda x: f"{x:.2f} años" if isinstance(x, (int, float, np.floating)) else x)

display(
    tabla.style.hide(axis="index").set_properties(
        subset=["interpretacion"],
        **{"white-space": "normal", "text-align": "left"}
    )
)

### Graficos de casos especiales

Estos graficos muestran los casos borde de la tabla. Se separan en varios paneles porque algunos crecen mucho y otros quedan cerca de cero. La linea roja `P = 1` sirve como referencia: por debajo de ese valor, una poblacion real ya no se puede interpretar como individuos completos.

In [ ]:
casos_grafico = [
    {"caso": "r = 0", "P0": 100, "r": 0, "t_final": 20},
    {"caso": "r < 0", "P0": 100, "r": -0.15, "t_final": 40},
    {"caso": "P0 = 0", "P0": 0, "r": 0.15, "t_final": 20},
    {"caso": "P0 < 0", "P0": -100, "r": 0.15, "t_final": 20},
    {"caso": "P0 = 1", "P0": 1, "r": 0.15, "t_final": 20},
    {"caso": "poblacion no entera", "P0": 100, "r": 0.15, "t_final": 5},
    {"caso": "r muy grande", "P0": 100, "r": 0.60, "t_final": 20},
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.ravel()

for ax, caso in zip(axes, casos_grafico):
    t, P = resolver_malthus(P0=caso["P0"], r=caso["r"], t_final=caso["t_final"])
    ax.plot(t, P, linewidth=2, label=caso["caso"])
    ax.axhline(0, color="black", linewidth=1)
    ax.axhline(1, color="tab:red", linestyle=":", linewidth=1.6, label="P = 1")
    ax.scatter([0], [caso["P0"]], color="gold", edgecolor="black", zorder=5, label="P0=" + str(caso["P0"]))
    ax.set_title(caso["caso"] + " | r=" + str(caso["r"]))
    ax.set_xlabel("Tiempo (años)")
    ax.set_ylabel("Poblacion")
    ax.legend(fontsize=8)

    if caso["caso"] == "r < 0":
        tiempo_menor_1 = np.log(1 / caso["P0"]) / caso["r"]
        ax.axvline(tiempo_menor_1, color="tab:red", linestyle="--", linewidth=1.4)
        ax.annotate("P < 1", xy=(tiempo_menor_1, 1), xytext=(tiempo_menor_1 + 2, 25), arrowprops={"arrowstyle": "->", "color": "tab:red"}, color="tab:red")

for ax in axes[len(casos_grafico):]:
    ax.axis("off")

fig.suptitle("Malthus - casos especiales")
plt.tight_layout()
plt.show()

### Supuestos del modelo

El modelo de Malthus supone:

- Recursos y espacio ilimitados.
- `r` constante en el tiempo.
- Sin migracion.
- Poblacion continua y grande.

Por eso funciona mejor como aproximacion de corto plazo y para poblaciones grandes. El modelo logistico de Verhulst corrige el crecimiento sin limite agregando una capacidad maxima del ambiente, pero no lo implementamos en este punto.

**Conclusion del punto 2.**

En Malthus, `r` define si la poblacion crece, queda constante o decrece, y tambien que tan rapido ocurre ese cambio. Si `r > 0`, el tiempo de duplicacion es `ln(2)/r`. Si `r < 0`, la poblacion se acerca a `0`, aunque el modelo continuo nunca llega exactamente.

`P0` solo escala la curva: cambiar la poblacion inicial cambia los valores absolutos, pero no el tiempo de duplicacion cuando `r` queda fijo.

El modelo es razonable para poblaciones grandes y en el corto plazo. Para poblaciones pequenas, valores no enteros o falta de recursos, deja de representar bien la realidad.

El decaimiento radioactivo del punto 3 es la misma ecuacion con tasa negativa (`r = -lambda`).